---
title: "Multiple Regression and Feature Engineering"
execute:
  enabled: true
jupyter: python3
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

# Load data
DATA_DIR = 'data'

## How much is a bathroom worth?

An Airbnb host is considering adding a second bathroom to their listing. How much more should they charge per night? In [Chapter 4](lec04-regression.qmd) we built a regression from the mean up and stopped at a single feature — bedrooms. That model can't answer the host's question. It doesn't even know bathrooms exist. Every prediction rests on bedrooms alone, and the bedrooms coefficient absorbs every pricing signal correlated with room count, bathrooms included.

Today we add bathrooms, then room type, then neighborhood. Along the way we discover that naively adding features can mislead: the coefficient on bedrooms shifts once bathrooms enter the model, and the coefficient on number of reviews turns out negative in a way no host should take at face value. We finish with **feature engineering** — polynomials, interactions, log transforms, and missingness indicators — which turns linear regression into a surprisingly expressive tool.

The roadmap: add numeric features (bathrooms), then categorical features (room type via one-hot encoding), then engineered features (polynomials, interactions, log transforms). At each step the model stays linear; we just expand the set of features.

## The data

In [ ]:
listings = pd.read_csv(f'{DATA_DIR}/airbnb/listings.csv', low_memory=False)

cols = ['id', 'name', 'description', 'price', 'bedrooms', 'bathrooms', 'beds',
        'room_type', 'neighbourhood_group_cleansed', 'accommodates',
        'number_of_reviews', 'cleaning_fee']
n_raw = len(listings)
df = listings[cols].dropna(subset=['price', 'bedrooms', 'bathrooms']).copy()
df = df.rename(columns={'neighbourhood_group_cleansed': 'borough'})

# Price is already a clean integer column (no $ signs); filter to reasonable range
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df = df[df['price'].between(10, 500)].reset_index(drop=True)
prices = df['price']

print(f"Raw listings:                 {n_raw:,}")
print(f"After dropping missing price/bedrooms/bathrooms and clipping price to $10–$500:")
print(f"  Working with {len(df):,} listings ({n_raw - len(df):,} dropped)")
df[['name', 'price', 'bedrooms', 'bathrooms', 'room_type', 'borough']].head()

# Multiple regression with numeric features

## Adding bathrooms

In Chapter 4, simple regression gave us price as a function of bedrooms alone. But a 2-bedroom apartment with 2 bathrooms is worth more than the same apartment with 1 bathroom. Let's add bathrooms as a second feature.

In [ ]:
# Simple regression (from Chapter 4)
model_simple = LinearRegression()
model_simple.fit(df[['bedrooms']], prices)
print(f"Simple regression:   price = {model_simple.intercept_:.0f} + {model_simple.coef_[0]:.0f} × bedrooms")
print(f"  R² = {model_simple.score(df[['bedrooms']], prices):.3f}")
print()

# Multiple regression: price ~ bedrooms + bathrooms
model_two = LinearRegression()
model_two.fit(df[['bedrooms', 'bathrooms']], prices)
print(f"Multiple regression: price = {model_two.intercept_:.0f} + {model_two.coef_[0]:.0f} × bedrooms + {model_two.coef_[1]:.0f} × bathrooms")
print(f"  R² = {model_two.score(df[['bedrooms', 'bathrooms']], prices):.3f}")

Adding bathrooms improves $R^2$. The intercept and the bedrooms coefficient both changed — we'll return to why shortly. First we need a compact way to talk about "the features" as a single object, and then a geometric reading of why $R^2$ went up at all.

## The feature matrix

Chapter 4 worked with one feature at a time — a single column vector $x$ of bedrooms, paired with the column of ones for the intercept. Multiple regression wants many features, and writing them out one at a time gets unwieldy. Stack them into a matrix instead.

You already know what a feature matrix looks like: it is the numeric part of a pandas DataFrame. Each row is one listing, each column is one feature, and each cell holds a number. Calling `df[['bedrooms', 'bathrooms']].values` converts that chunk of the DataFrame into a NumPy matrix — that matrix *is* $X$, just with a mathematical name.

:::{.callout-important}
## Definition: Feature matrix (design matrix)

With $p$ features measured on $n$ listings, stack the feature vectors as columns of an $n \times p$ **feature matrix** $X$ (statistics textbooks call the same object the **design matrix**; we use the terms interchangeably). Each *row* of $X$ is one listing's features; each *column* is one feature across all listings. A linear model with this matrix is
$$\hat{y} = X\beta,$$
where $\beta \in \mathbb{R}^p$ is a vector of coefficients, one per column of $X$. By convention we include a column of ones in $X$ so that $\beta_0$ (the intercept) sits alongside the other coefficients.
:::

For the two-feature model above, $X$ has three columns: the all-ones column, `bedrooms`, and `bathrooms`. The prediction for listing $i$ is
$$\hat{y}_i = \beta_0 \cdot 1 + \beta_1 \cdot \text{bedrooms}_i + \beta_2 \cdot \text{bathrooms}_i,$$
and stacking those $n$ predictions gives $\hat{y} = X\beta$.

Every prediction $\hat{y} = X\beta$ is a linear combination of the columns of $X$ — the span (also called the **column space**) from Chapter 4, now with more vectors. With one feature plus an intercept the span was a 2D plane inside $\mathbb{R}^n$; adding bathrooms as a third column expands it to a three-dimensional set of reachable predictions. More columns mean a larger span, so the projection of $y$ can only get closer. Training $R^2$ can only increase (or stay the same) when you add a feature.

## Normal equations

The key insight carries over from Chapter 4: $\hat{y}$ sits as close to $y$ as the feature columns allow, which happens exactly when the residual is perpendicular to every feature direction. With one feature we wrote that idea as two scalar conditions — $\mathbf{1}^T \epsilon = 0$ (residuals sum to zero) and $x^T \epsilon = 0$ (no linear pattern between the feature and the residuals). With many features we want to say the same thing all at once, and linear algebra gives us a clean way to do it: stack every perpendicularity condition into a single matrix equation.

Write $X$ for the $n \times p$ feature matrix (including the intercept column of ones). The orthogonality conditions become:

$$X^T \epsilon = 0 \quad \text{where } \epsilon = y - X\beta.$$

Substitute $\epsilon = y - X\beta$ and distribute:

$$X^T(y - X\beta) = 0 \;\Longrightarrow\; X^T y - X^T X \beta = 0 \;\Longrightarrow\; X^T X \beta = X^T y.$$

When $X$ has linearly independent columns, $X^T X$ is invertible and we can solve for $\beta$:

$$\beta = (X^T X)^{-1} X^T y.$$

:::{.callout-important}
## Definition: Normal equations

The **normal equations** $X^TX\beta = X^Ty$ give the least squares coefficients. Each row of $X^T \epsilon = 0$ says "the residual has zero inner product with one column of $X$" — orthogonality to every feature simultaneously.
:::

In [ ]:
# Build design matrix with intercept
X_manual = np.column_stack([
    np.ones(len(df)),
    df['bedrooms'].values,
    df['bathrooms'].values
])

# Normal equations: beta = (X^T X)^{-1} X^T y
beta_normal = np.linalg.solve(X_manual.T @ X_manual, X_manual.T @ prices.values)

print("Normal equations:  intercept = {:.2f}, bedrooms = {:.2f}, bathrooms = {:.2f}".format(*beta_normal))
print("sklearn:           intercept = {:.2f}, bedrooms = {:.2f}, bathrooms = {:.2f}".format(
    model_two.intercept_, model_two.coef_[0], model_two.coef_[1]))

The two approaches agree. We use `np.linalg.solve` rather than computing $(X^TX)^{-1}$ explicitly — matrix inversion is numerically unstable for large or ill-conditioned problems.

In [ ]:
# Verify orthogonality: residuals have zero dot product with every column
y_hat_two = model_two.predict(df[['bedrooms', 'bathrooms']])
residuals_two = prices.values - y_hat_two

dot_ones = np.dot(residuals_two, np.ones(len(df)))
dot_beds = np.dot(residuals_two, df['bedrooms'].values)
dot_baths = np.dot(residuals_two, df['bathrooms'].values)

print(f"Residuals dot (ones):      {dot_ones:.4f}  (≈ 0)")
print(f"Residuals dot (bedrooms):  {dot_beds:.4f}  (≈ 0)")
print(f"Residuals dot (bathrooms): {dot_baths:.4f}  (≈ 0)")

## Interpreting coefficients

The coefficient on bedrooms changed from the simple regression. Why?

In [ ]:
print(f"Simple regression:   bedrooms coefficient = ${model_simple.coef_[0]:.0f}")
print(f"Multiple regression: bedrooms coefficient = ${model_two.coef_[0]:.0f}")
print(f"                     bathrooms coefficient = ${model_two.coef_[1]:.0f}")

Bedrooms and bathrooms travel together: apartments with more bedrooms tend to have more bathrooms too. Simple regression on bedrooms alone has no way to split the gain — it hands all the price difference to bedrooms, since that is the only variable it can see, and the coefficient absorbs everything **correlated** with bedrooms, bathrooms included. Multiple regression asks a sharper question: *among listings with the same number of bathrooms, how does price change with bedrooms?* That comparison is what "holding bathrooms constant" means.

:::{.callout-tip}
## Think About It

When you compare two listings with the same number of bathrooms, the one with an extra bedroom costs about \$46 more (according to the two-feature model). When you compare two listings with the same number of bedrooms, the one with an extra bathroom costs about \$12 more. Notice that the simple regression said each bedroom was worth \$49 — the coefficient dropped when we controlled for bathrooms. These coefficients measure **association**, not causation — we'll revisit that distinction in [Chapter 18](lec18-causal.qmd).
:::

# One-hot encoding and categorical features

## The problem

Room type is one of the strongest predictors of price — entire apartments cost more than private rooms. But room type is a string:

In [ ]:
print("Room types:", df['room_type'].unique())
print()
print("Average price by room type:")
print(df.groupby('room_type')['price'].mean().round(0).sort_values(ascending=False))

You cannot multiply "Entire home/apt" by a coefficient. Regression requires numbers.

:::{.callout-tip}
## Think About It

How would you convert room type to numbers so a regression model can use it? A first instinct is to assign each category a single integer — say, entire home = 2, private room = 1, shared room = 0. Think before reading on: what would a linear model actually do with those numbers, and why is that a problem?
:::

## One-hot encoding

The solution: create one binary (0/1) column per category, called an **indicator variable** (or dummy variable). A listing that is an "Entire home/apt" gets a 1 in that column and 0 in the others. The function `pd.get_dummies()` does the work.

In [ ]:
# One-hot encode room type
room_dummies = pd.get_dummies(df['room_type'], prefix='room_type')
room_dummies.head()

There is a subtlety. Add the three dummy columns together, and you get a column of all ones — exactly the intercept column $\mathbf{1}$ that is already in the model. That means one of the dummies can be written as the intercept minus the other two: they form a **linearly dependent** set.

:::{.callout-important}
## Definition: Linear dependence

A set of vectors $v_1, \ldots, v_k$ is **linearly dependent** if one of them can be written as a linear combination of the others. Equivalently, there exist scalars $c_1, \ldots, c_k$, not all zero, such that
$$c_1 v_1 + c_2 v_2 + \cdots + c_k v_k = 0.$$
A set that is not linearly dependent is **linearly independent**.
:::

The room-type dummies illustrate the definition concretely. Let $e, p, s$ be the Entire-home, Private-room, and Shared-room indicator columns. Every listing is in exactly one category, so for every row
$$e_i + p_i + s_i = 1 = \mathbf{1}_i,$$
and rearranging gives $\mathbf{1} - e - p - s = 0$ — a non-trivial linear combination of $\{\mathbf{1}, e, p, s\}$ that equals the zero vector. The four columns are linearly dependent.

Adding a linearly dependent column doesn't grow the span — it just creates a collinearity, and the normal equations no longer have a unique solution ($X^TX$ is singular, so $(X^TX)^{-1}$ does not exist). The fix: drop one category, called the **reference level**, so the remaining dummy columns are linearly independent of the intercept.

:::{.callout-important}
## Definition: Reference level

When one-hot encoding a categorical variable with $k$ categories, we create $k - 1$ binary columns and drop one. The dropped category is the **reference level**. Its baseline is absorbed into the intercept; every other coefficient measures the difference relative to this baseline. Using `drop_first=True` in `pd.get_dummies()` drops the first category alphabetically.
:::

In [ ]:
# One-hot encode with reference level dropped
room_dummies_ref = pd.get_dummies(df['room_type'], prefix='room_type', drop_first=True)
print("Columns after dropping reference level:")
print(room_dummies_ref.columns.tolist())
print(f"\nReference level (dropped): {sorted(df['room_type'].unique())[0]}")
room_dummies_ref.head()

## Interpreting coefficients relative to the reference

With "Entire home/apt" as the reference level, the intercept absorbs the baseline price for entire homes. The coefficient on `room_type_Private room` tells you how much less a private room costs relative to an entire home, holding all other features constant.

In [ ]:
# Regression with room type only
X_room = room_dummies_ref.astype(float)
model_room = LinearRegression().fit(X_room, prices)

print(f"Intercept (Entire home/apt baseline): ${model_room.intercept_:.0f}")
for name, coef in zip(X_room.columns, model_room.coef_):
    print(f"  {name}: ${coef:+.0f}")

The intercept is approximately the mean price of entire homes. The private room coefficient is negative — private rooms cost less than entire homes. The shared room coefficient is even more negative. Each coefficient is a price *difference* relative to the reference category.

## The full model

Now we combine everything: bedrooms, bathrooms, and room type. We prepare all features in one design matrix — bedrooms and bathrooms stay as numeric columns, and room type is one-hot encoded with `drop_first=True` so the "Entire home/apt" category becomes the reference level. The intercept will absorb that baseline; the other room-type coefficients measure price differences relative to it.

In [ ]:
# Numeric features stay as-is; room type becomes k-1 dummies
num_features = df[['bedrooms', 'bathrooms']]
cat_features = pd.get_dummies(df[['room_type']], drop_first=True)
X_full = pd.concat([num_features, cat_features], axis=1).astype(float)

In [ ]:
# Fit the full model and collect coefficients
model_full = LinearRegression().fit(X_full, prices)
r2_full = model_full.score(X_full, prices)

coef_table = pd.DataFrame({
    'feature': list(X_full.columns),
    'coefficient': model_full.coef_.round(2)
}).sort_values('coefficient', key=abs, ascending=False)

print(f"R² = {r2_full:.3f}")
print(f"Intercept: ${model_full.intercept_:.2f}\n")
print(coef_table.to_string(index=False))

The model finally answers the opening question: a bathroom is associated with roughly \$22 more per night, holding bedrooms and room type constant. An extra bedroom, once bathrooms and room type are controlled for, is associated with about \$33 more. Private rooms cost substantially less than entire homes, even after adjusting for size, and shared rooms less still.

To see the coefficients at a glance, we plot them as a horizontal bar chart with dollar labels on each bar.

In [ ]:
# Visualize coefficients with dollar labels
fig, ax = plt.subplots(figsize=(8, 4))
coef_sorted = coef_table.sort_values('coefficient').copy()

# Rename the raw pandas dummy names to human-readable labels
label_map = {
    'bedrooms': 'Bedrooms',
    'bathrooms': 'Bathrooms',
    'room_type_Private room': 'Private room\n(vs. entire home)',
    'room_type_Shared room': 'Shared room\n(vs. entire home)',
}
coef_sorted['feature'] = coef_sorted['feature'].map(lambda s: label_map.get(s, s))

colors = ['#d62728' if c < 0 else '#2171b5' for c in coef_sorted['coefficient']]
bars = ax.barh(coef_sorted['feature'], coef_sorted['coefficient'],
               color=colors, edgecolor='white')
ax.axvline(0, color='gray', linewidth=1)
ax.set_xlabel('Coefficient (association with price, $)')
ax.set_title('Multiple regression coefficients: bedrooms + bathrooms + room type')

# Annotate each bar with its dollar value
for bar, val in zip(bars, coef_sorted['coefficient']):
    offset = 2 if val >= 0 else -2
    ha = 'left' if val >= 0 else 'right'
    ax.text(val + offset, bar.get_y() + bar.get_height() / 2,
            f'${val:+.0f}', va='center', ha=ha, fontsize=10)

ax.margins(x=0.15)
plt.tight_layout()
plt.show()

## The reviews puzzle

What happens when we add `number_of_reviews` to the model? Before looking at the coefficient, compare $R^2$ with and without the new feature.

In [ ]:
# Add number of reviews — drop rows with missing review counts first
df_reviews = df.dropna(subset=['number_of_reviews']).copy()
X_reviews = pd.concat([
    df_reviews[['bedrooms', 'bathrooms', 'number_of_reviews']],
    pd.get_dummies(df_reviews[['room_type']], drop_first=True)
], axis=1).astype(float)
y_reviews = df_reviews['price']

# Refit the baseline model on the same rows so R² values are comparable
X_base_reviews = pd.concat([
    df_reviews[['bedrooms', 'bathrooms']],
    pd.get_dummies(df_reviews[['room_type']], drop_first=True)
], axis=1).astype(float)
r2_base_reviews = LinearRegression().fit(X_base_reviews, y_reviews).score(X_base_reviews, y_reviews)

model_reviews = LinearRegression().fit(X_reviews, y_reviews)
r2_reviews = model_reviews.score(X_reviews, y_reviews)

print(f"Without reviews: R² = {r2_base_reviews:.4f}")
print(f"With reviews:    R² = {r2_reviews:.4f}")
print(f"Improvement:     {r2_reviews - r2_base_reviews:+.5f}")

The R² is essentially identical to four decimal places — adding reviews buys us nothing in predictive accuracy.

In [ ]:
print("Coefficients with reviews included:")
for name, coef in zip(X_reviews.columns, model_reviews.coef_):
    print(f"  {name:30s}  {coef:+.2f}")

The coefficient on `number_of_reviews` is *negative*. The association is real: listings with more reviews do tend to have lower prices in this data, holding other features constant. But reading it as a recommendation — "get more reviews to lower your price" — is wrong. Two mechanisms work together to produce the negative sign, and neither is a causal effect of reviews on price:

- **Reverse causation** — the response $y$ is influencing the feature $x$, not the other way around. Price influences how often a listing sells, bookings drive review counts, so reviews sit downstream of price. A \$60/night studio might host 200 guests a year; a \$350/night loft might host 30. More guests means more reviews.
- **Omitted variables** — listing age, host activity, and segment (budget vs. luxury) drive both review counts *and* prices for reasons that have nothing to do with a price-to-reviews link. Old, casual, lower-end listings accumulate more reviews and charge less; neither direction is a causal channel between the two.

The regression faithfully reports the downward association but has no window into which of these stories is generating it. An analyst who stops at the coefficient — or worse, recommends a host strategy based on it — has confused association with causation.

:::{.callout-tip}
## Think About It

When a coefficient surprises you, ask: *could the response plausibly influence this feature?* Here, high price discourages bookings, and fewer bookings mean fewer reviews — so $y$ (price) causally affects $x$ (reviews), not the other way around. Multiple regression cannot untangle this from observational data alone. Could you design a study that separates the causal direction? What data would you need beyond this cross-sectional snapshot? We develop such tools in [Chapter 18](lec18-causal.qmd).
:::

# Feature engineering: transforming X

## The big idea

Every model we've fit so far is $\hat{y} = X\beta$ — linear in the parameters $\beta$. The features $X$ can be anything: raw columns, squared columns, products of columns, or columns derived from text. The mantra captures the point: **"linear in parameters, not in features."** The creative work happens *before* the fit, in the choice of which columns to build.

This section works through three moves. **Interaction terms** let the slope depend on another variable. **Polynomial features** let the slope bend. **Missing-value indicators** turn an awkward data-cleaning problem into a feature. The next section then reshapes the response with a log transform.

## Interaction terms

Is an extra bedroom associated with the same price increase in Manhattan and the Bronx? So far, our model uses a single bedrooms coefficient everywhere. An **interaction term** lets the association between one feature and the response depend on another:

$$\hat{y} = \beta_0 + \beta_1 \cdot \text{bedrooms} + \beta_2 \cdot \mathbb{1}_{\text{Manhattan}} + \beta_3 \cdot (\text{bedrooms} \times \mathbb{1}_{\text{Manhattan}})$$

The coefficient $\beta_3$ captures how much *more* (or less) a bedroom is worth in Manhattan compared to the reference borough.

In [ ]:
# Pooled model: bedrooms + borough dummies (shared slope, borough-specific intercepts)
borough_dummies = pd.get_dummies(df['borough'], drop_first=True).astype(float)
X_pooled = pd.concat([df[['bedrooms']], borough_dummies], axis=1)
model_pooled = LinearRegression().fit(X_pooled, prices)

# Interaction model: add bedrooms × borough columns
interactions = borough_dummies.multiply(df['bedrooms'], axis=0)
interactions.columns = [f'bedrooms_x_{col}' for col in interactions.columns]
X_interact_only = pd.concat([df[['bedrooms']], borough_dummies, interactions], axis=1)
model_interact_only = LinearRegression().fit(X_interact_only, prices)

# The reference borough is the one that was dropped by drop_first=True
ref_borough = sorted(df['borough'].unique())[0]
shared_slope = model_pooled.coef_[0]
print(f"Pooled model: shared bedroom slope = ${shared_slope:.0f}/BR across all boroughs")
print(f"Reference borough (absorbed into intercept): {ref_borough}")

With both models in hand, we can draw one line per borough under each. Under the pooled model every borough uses the same slope; under the interaction model every borough uses its own.

In [ ]:
# Draw per-borough lines implied by each model
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4), sharey=True)
palette = sns.color_palette('Set2', n_colors=df['borough'].nunique())
borough_order = [ref_borough] + list(borough_dummies.columns)
color_map = dict(zip(borough_order, palette))

def borough_intercept_pooled(borough):
    intercept = model_pooled.intercept_
    if borough != ref_borough:
        idx = list(borough_dummies.columns).index(borough) + 1
        intercept = intercept + model_pooled.coef_[idx]
    return intercept

def borough_line_interact(borough):
    intercept = model_interact_only.intercept_
    slope = model_interact_only.coef_[0]
    if borough != ref_borough:
        nb = len(borough_dummies.columns)
        idx_dummy = list(borough_dummies.columns).index(borough) + 1
        idx_inter = 1 + nb + list(borough_dummies.columns).index(borough)
        intercept = intercept + model_interact_only.coef_[idx_dummy]
        slope = slope + model_interact_only.coef_[idx_inter]
    return intercept, slope

for ax, title, kind in [
    (axes[0], 'Without interaction (parallel lines, shared slope)', 'pooled'),
    (axes[1], 'With interaction (different slopes per borough)', 'interact')
]:
    for borough, group in df.groupby('borough'):
        color = color_map[borough]
        ax.scatter(group['bedrooms'], group['price'], alpha=0.07, s=6, color=color)
        if len(group) > 50:
            x_range = np.array([0.0, group['bedrooms'].max()])
            if kind == 'pooled':
                intercept = borough_intercept_pooled(borough)
                slope = shared_slope
            else:
                intercept, slope = borough_line_interact(borough)
            ax.plot(x_range, intercept + slope * x_range, color=color, linewidth=2.5,
                    label=f'{borough} (${slope:.0f}/BR)')
    ax.set_xlabel('Bedrooms')
    ax.set_ylabel('Price ($)')
    ax.set_title(title)
    ax.legend(fontsize=8, loc='upper left')
    ax.set_xlim(-0.5, 6.5)
    ax.set_ylim(0, 500)

plt.tight_layout()
plt.show()

In the left panel every borough shares the same slope — the pooled model forces one bedroom coefficient for the whole city, and the only room for variation across boroughs is the intercept. In the right panel the interaction model lets each borough set its own slope, and the slopes differ substantially. An extra bedroom in Manhattan or Brooklyn adds much more to the predicted price than the same bedroom in the Bronx.

In [ ]:
# Compare R² across the two models
r2_pooled = model_pooled.score(X_pooled, prices)
r2_interact = model_interact_only.score(X_interact_only, prices)
print(f"Pooled (borough intercepts only): R² = {r2_pooled:.4f}")
print(f"With bedrooms × borough:          R² = {r2_interact:.4f}")

The interaction model also nests the room-type and bathrooms features from earlier. Refitting with the full feature set plus interactions gives our richest model so far.

In [ ]:
# Full model + bedrooms × borough interactions for the recap R² table
X_interact_full = pd.concat([X_full, borough_dummies, interactions], axis=1).astype(float)
model_interact = LinearRegression().fit(X_interact_full, prices)
r2_interact_full = model_interact.score(X_interact_full, prices)
print(f"Full features + borough + interactions: R² = {r2_interact_full:.4f}  ({X_interact_full.shape[1]} features)")

A model without interaction terms hides borough-level heterogeneity inside a single pooled slope. Interactions let the model say what every real-estate agent already knows: a bedroom is not worth the same dollars everywhere.

## Polynomial features

Interactions multiplied features together to let slopes depend on a category. Polynomials multiply a feature by *itself* to let the slope bend.

Does price increase at the same rate per bedroom, or does the relationship flatten? A linear model forces a constant slope. Adding **polynomial features** — $x$, $x^2$, $x^3$ — lets the model capture curvature while staying linear in $\beta$:

$$\hat{y} = \beta_0 + \beta_1 x + \beta_2 x^2$$

The equation is nonlinear in $x$ but linear in $(\beta_0, \beta_1, \beta_2)$. The function `PolynomialFeatures` from scikit-learn generates these columns.

In [ ]:
X_bedrooms = df[['bedrooms']]

# Fit polynomials of increasing degree
for degree in [1, 2, 3]:
    # LinearRegression adds its own intercept; we skip the bias column here
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly = poly.fit_transform(X_bedrooms)
    model_poly = LinearRegression().fit(X_poly, prices)
    r2 = model_poly.score(X_poly, prices)
    print(f"Degree {degree}: R² = {r2:.4f}  ({X_poly.shape[1]} features)")

The degree-2 model captures a gentle price plateau at higher bedroom counts. But what happens at higher degrees?

## The polynomial overfitting parade

Let's fit polynomials from degree 1 through 6 and see what each predicts across the range of observed listings.

:::{.callout-tip}
## Think About It (before looking at the plot)

Degree 1 is a straight line. Degree 2 above already bent into a gentle plateau. Before fitting degrees 3–6: commit to a prediction. Will training $R^2$ keep climbing? What will a degree-6 fit look like *inside* the observed range (0–5 bedrooms)? What about at 9 bedrooms, beyond anything the data covers? Write down your guess, then check it against the plot below.
:::

In [ ]:
MAX_DEGREE = 6
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4))
max_obs = float(df['bedrooms'].max())

# Left: polynomial fits extrapolated
ax = axes[0]
ax.scatter(df['bedrooms'], prices, alpha=0.05, s=5, color='gray', label='data')
x_extrap = np.linspace(0, 10, 300).reshape(-1, 1)

# Shade the region beyond the observed bedroom counts
ax.axvspan(max_obs, 10, alpha=0.15, color='red', label='Extrapolation region')

colors_poly = plt.cm.viridis(np.linspace(0, 0.9, MAX_DEGREE))
r2_list = []
top_degree_pred = None

for degree in range(1, MAX_DEGREE + 1):
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_p = poly.fit_transform(X_bedrooms)
    m = LinearRegression().fit(X_p, prices)
    r2_list.append(m.score(X_p, prices))
    pred = m.predict(poly.transform(x_extrap))
    if degree == MAX_DEGREE:
        top_degree_pred = pred
    if degree in [1, 2, 5, MAX_DEGREE]:
        ax.plot(x_extrap, pred, color=colors_poly[degree-1], linewidth=2,
                label=f'Degree {degree}')

ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Bedrooms')
ax.set_ylabel('Predicted price ($)')
ax.set_title('R² says higher degree is better. Do you agree?')
ax.legend(fontsize=9, loc='upper left')
ax.set_ylim(-300, 600)

# Annotate the highest-degree curve inside the extrapolation region
ax.annotate(f'Degree {MAX_DEGREE} swings\nwildly here',
            xy=(9.3, top_degree_pred[-10]),
            xytext=(7.2, -220),
            fontsize=9, color='#b00020',
            arrowprops=dict(arrowstyle='->', color='#b00020', lw=1.2))

# Right: R² by degree
ax = axes[1]
bars = ax.bar(range(1, MAX_DEGREE + 1), r2_list, color=colors_poly, edgecolor='white')
for i, r2_val in enumerate(r2_list):
    ax.text(i + 1, r2_val + 0.003, f'{r2_val:.3f}', ha='center', fontsize=8)
ax.set_xlabel('Polynomial degree')
ax.set_ylabel('Training R²')
ax.set_title('R² climbs — but the gains shrink fast')

plt.tight_layout()
plt.show()

Low-degree polynomials produce reasonable curves. High-degree polynomials oscillate at the edges of the observed range and extrapolate nonsensically beyond it. Inside the observed data the degree-6 curve already wobbles noticeably, and inside the shaded extrapolation region its predictions swing into territory no sensible host would accept. Polynomial features fit training data well but extrapolate according to their mathematical form, not the underlying phenomenon.

Meanwhile, training $R^2$ keeps inching up — from about 0.16 at degree 1 to 0.27 at degree 6 — while each additional degree buys less and less. Degrees beyond 6 add essentially nothing to training $R^2$ while making the extrapolation wilder. We'll develop formal tools to detect overfitting in [Chapter 6](lec06-validation.qmd).

In [ ]:
# Predictions at specific bedroom counts
print("Predicted price by bedroom count:")
degrees_to_show = [1, 2, 5, MAX_DEGREE]
print(f"{'Bedrooms':>10}", end='')
for d in degrees_to_show:
    print(f"{'Degree '+str(d):>12}", end='')
print()

for beds in [1, 3, 5, 7, 9]:
    print(f"{beds:>10}", end='')
    for d in degrees_to_show:
        poly = PolynomialFeatures(degree=d, include_bias=False)
        X_p = poly.fit_transform(X_bedrooms)
        m = LinearRegression().fit(X_p, prices)
        pred = m.predict(poly.transform([[beds]]))[0]
        print(f"      ${pred:>5.0f}", end='')
    print()

:::{.callout-note}
## Parametric extrapolation in the real world

In early 2020, the influential IHME COVID-19 model forecast U.S. deaths by fitting a Gaussian curve — a symmetric bell shape — to the daily death counts. Because the parametric form assumed deaths would decline as fast as they rose, the model repeatedly projected the pandemic was about to end even as deaths were still climbing (Jewell et al., *JAMA*, 2020). The assumed shape, not the data, was driving the predictions. The broader lesson applies to polynomials too: any rigid parametric model extrapolates according to its mathematical form, not the underlying phenomenon.
:::

## Missing values as features

Polynomials and interactions both built new columns out of existing features. The last move in this section builds a column out of the *absence* of a feature.

A missing `cleaning_fee` carries information. Hosts who leave it blank might bundle cleaning into the nightly rate, might be casual operators who skipped optional fields, or might expect guests to tidy up. Whatever the mechanism, listings with no listed fee are systematically cheaper, and we want the model to see it.

In [ ]:
# cleaning_fee is stored as a numeric dollar amount (already parsed);
# a substantial fraction of listings leave it blank.
n_missing_fee = df['cleaning_fee'].isna().sum()
pct_missing_fee = 100 * n_missing_fee / len(df)
print(f"Listings with missing cleaning fee: {n_missing_fee:,} ({pct_missing_fee:.1f}%)")
print()

# Median price split by whether the fee is present
median_missing = df[df['cleaning_fee'].isna()]['price'].median()
median_present = df[df['cleaning_fee'].notna()]['price'].median()
print(f"Median price when cleaning fee is missing: ${median_missing:.0f}")
print(f"Median price when cleaning fee is present: ${median_present:.0f}")

Listings with a missing cleaning fee are noticeably cheaper on average — the missingness isn't random noise, it's tracking something real about the listing. The approach: impute the missing numeric value with *any* constant, then add a binary `fee_missing` column. The model can then learn whether "missing" predicts a different price from any observed fee.

The choice of imputation constant does not affect predictions. Imputing with $c$ instead of zero shifts $\hat{\beta}_{\text{missing}}$ by exactly $-c \cdot \hat{\beta}_{\text{fee}}$, leaving $\hat{y}$ unchanged for every row. To see why: the imputed-with-$c$ column equals the imputed-with-zero column plus $c$ times the indicator, so the two designs span the same column space and produce the same projection. We pick **zero** because it gives the indicator coefficient a clean interpretation — $\hat{\beta}_{\text{missing}}$ directly measures the total contribution from a missing-fee listing, with no $\hat{\beta}_{\text{fee}} \cdot c$ offset to unwind.

In [ ]:
# Impute the cleaning fee with zero and record which rows were filled in.
df['cleaning_fee_imputed'] = df['cleaning_fee'].fillna(0)
df['fee_missing'] = df['cleaning_fee'].isna().astype(int)

print("Sample rows:")
print(df[['cleaning_fee', 'cleaning_fee_imputed', 'fee_missing']].head(10))

In [ ]:
# Verify: imputing with c=0 vs c=50 gives identical predictions
fee_0  = df['cleaning_fee'].fillna(0)
fee_50 = df['cleaning_fee'].fillna(50)
indicator = df['fee_missing']

X_c0  = pd.concat([X_full, fee_0.rename('fee'), indicator], axis=1).astype(float)
X_c50 = pd.concat([X_full, fee_50.rename('fee'), indicator], axis=1).astype(float)

m0  = LinearRegression().fit(X_c0, prices)
m50 = LinearRegression().fit(X_c50, prices)

print("Max prediction difference:", np.abs(m0.predict(X_c0) - m50.predict(X_c50)).max())
print(f"c=0:  β_fee={m0.coef_[-2]:+.4f},  β_missing={m0.coef_[-1]:+.2f}")
print(f"c=50: β_fee={m50.coef_[-2]:+.4f},  β_missing={m50.coef_[-1]:+.2f}")
print(f"Shift: β_missing changed by {m50.coef_[-1] - m0.coef_[-1]:+.2f}  "
      f"= -50 × β_fee = {-50 * m0.coef_[-2]:+.2f}")

The coefficients shift but the predictions are identical — the column space is the same. With zero imputation, $\hat{\beta}_{\text{missing}}$ reads directly as the price premium for a missing-fee listing.

Does the indicator carry its own weight, beyond the imputed value?

In [ ]:
# Baseline: the full model without any cleaning-fee information
X_base = X_full
m_base = LinearRegression().fit(X_base, prices)
r2_base = m_base.score(X_base, prices)

# Add the imputed fee by itself
X_fee = pd.concat([X_full, df[['cleaning_fee_imputed']]], axis=1).astype(float)
m_fee = LinearRegression().fit(X_fee, prices)
r2_fee = m_fee.score(X_fee, prices)

# Add the imputed fee and the missingness indicator
X_fee_ind = pd.concat([X_full, df[['cleaning_fee_imputed', 'fee_missing']]], axis=1).astype(float)
m_fee_ind = LinearRegression().fit(X_fee_ind, prices)
r2_fee_ind = m_fee_ind.score(X_fee_ind, prices)

print(f"Baseline (no fee):            R² = {r2_base:.4f}")
print(f"+ imputed cleaning fee:       R² = {r2_fee:.4f}")
print(f"+ imputed fee + fee_missing:  R² = {r2_fee_ind:.4f}")
print()
beta_fee = m_fee_ind.coef_[-2]
beta_missing = m_fee_ind.coef_[-1]
median_fee = df['cleaning_fee'].median()
print(f"cleaning_fee_imputed coefficient: ${beta_fee:+.2f} per $1 of fee")
print(f"fee_missing coefficient:          ${beta_missing:+.2f}")
print()
print(f"Median observed cleaning fee: ${median_fee:.0f}")
print(f"Fee contribution for a median-fee listing: ${beta_fee * median_fee:+.2f}")
print(f"Fee contribution for a missing-fee listing: ${beta_missing:+.2f}")

The imputed fee pushes $R^2$ up sharply — the fee itself is a strong price signal. Adding the `fee_missing` indicator contributes a smaller additional bump. Now the key move: read the **total contribution** each row gets from the two new columns.

- A listing with a missing fee has `cleaning_fee_imputed = 0` and `fee_missing = 1`, so its combined contribution is $\hat{\beta}_{\text{fee}} \cdot 0 + \hat{\beta}_{\text{missing}} \cdot 1 = \hat{\beta}_{\text{missing}} \approx \$25.87$.
- A listing with the median observed fee ($50) has `cleaning_fee_imputed = 50` and `fee_missing = 0`, so its combined contribution is $\hat{\beta}_{\text{fee}} \cdot 50 \approx \$31.53$.

The missing-fee listings are predicted about \$5–6 cheaper than a typical-fee listing, holding bedrooms, bathrooms, and room type fixed. That gap is the "missing carries information" signal: the model has learned something from the *pattern* of missingness that raw imputation alone can't capture.

:::{.callout-warning}
## Don't impute blindly

Imputation replaces missing values with plausible substitutes. Without a missingness indicator, the model cannot distinguish real values from imputed ones. Always pair imputation with an indicator column when missingness might carry information — that is, when the data is MAR or MNAR (see [Chapter 3](lec03-munging.qmd)).
:::


# Log transforms: transforming y

## From additive to multiplicative

The polynomial parade tried to capture curvature by piling on higher-order terms — and extrapolated into absurdity. The problem was the model's additive structure: each new term adds a correction, and nothing in the math keeps predictions positive or prevents them from swinging below zero.

A **log transform** attacks the same curvature from a different angle. Instead of letting $X$ grow into more columns, it reshapes $y$. The motivation is a simple observation about the bedroom premium: adding a bedroom to a luxury apartment is worth more in absolute dollars than adding a bedroom to a rundown studio. Large Airbnbs are scarce; the handful that can host a family reunion or a bachelor party command a disproportionate premium over the many cramped studios. The dollar response to "one more bedroom" *grows* with the base size, which is exactly the super-linear pattern a straight-line fit cannot match.

A log transform captures that pattern cleanly. On the log scale, a one-unit increase in bedrooms multiplies the price by a *constant factor*. Apply the same multiplier to a big Manhattan base and you get a big dollar jump; apply it to a small Bronx base and you get a small dollar jump — same proportional increase, very different dollar amounts.

In [ ]:
# Fit two models we can compare: a level model and a log model, each with borough
log_prices = np.log(prices)

X_level_b = pd.concat([df[['bedrooms']], borough_dummies], axis=1).astype(float)
model_level_b = LinearRegression().fit(X_level_b, prices)
r2_level_b = model_level_b.score(X_level_b, prices)

model_loglevel = LinearRegression().fit(X_level_b, log_prices)
r2_loglevel = model_loglevel.score(X_level_b, log_prices)

print(f"price      ~ bedrooms + borough: R² = {r2_level_b:.4f}")
print(f"log(price) ~ bedrooms + borough: R² = {r2_loglevel:.4f}  (comparing log-y to log-y)")
print(f"\nShared multiplier per bedroom (log model): e^{model_loglevel.coef_[0]:.3f}"
      f" = {np.exp(model_loglevel.coef_[0]):.3f}")

The two $R^2$ values measure different things (one fits price, one fits log price) so the bare comparison is not decisive. The visual is: on a log-scaled y-axis, the curvy cloud straightens out.

In [ ]:
# Visualize the log transform's effect using a semilogy y-axis
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4), sharex=True)
bedroom_grid = np.linspace(0, 6, 200)

# Left: level model overlaid on a semilogy plot.
# The fit is a straight line in dollars but curves on a log-y axis.
ax = axes[0]
ax.scatter(df['bedrooms'], prices, alpha=0.05, s=5, color='gray')
level_pred = model_simple.predict(bedroom_grid.reshape(-1, 1))
ax.plot(bedroom_grid, level_pred, 'r-', lw=2.5, label='level fit')
ax.set_yscale('log')
ax.set_xlabel('Bedrooms')
ax.set_ylabel('Price ($, log scale)')
ax.set_title('Level fit: curves on log-y axis')
ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
ax.legend(fontsize=9, loc='upper left')

# Right: log fit on a semilogy plot - straight line in log space
ax = axes[1]
ax.scatter(df['bedrooms'], prices, alpha=0.05, s=5, color='gray')
# Fit log(price) ~ bedrooms (single feature), then exponentiate to plot on price axis
model_log_simple = LinearRegression().fit(df[['bedrooms']], log_prices)
log_fit_price = np.exp(model_log_simple.predict(bedroom_grid.reshape(-1, 1)))
ax.plot(bedroom_grid, log_fit_price, 'r-', lw=2.5, label='log-level fit')
ax.set_yscale('log')
ax.set_xlabel('Bedrooms')
ax.set_ylabel('Price ($, log scale)')
ax.set_title('Log-level fit: straight on log-y axis')
ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
ax.legend(fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

On a log-scaled y-axis, the log-level fit is a straight line and the level fit visibly curves. The log model has matched the super-linear bedroom premium that the level model had to fake with a tilted straight line.

## Dollar increases differ by borough, percentage increases don't

The log-with-borough fit makes the Manhattan-vs-Bronx story numerical. Every borough gets the same multiplier per bedroom, but that multiplier applies to different base prices — so the *dollar* increase per bedroom ends up much larger in expensive boroughs.

In [ ]:
# Translate the log-level coefficients back into dollar increases per bedroom
# by borough. Each borough's base (0-bedroom) price differs; the multiplier
# per bedroom is shared across boroughs.
beta_bed_log = model_loglevel.coef_[0]
multiplier = np.exp(beta_bed_log)

rows = []
for i, borough in enumerate([ref_borough] + list(borough_dummies.columns)):
    if i == 0:
        log_base = model_loglevel.intercept_
    else:
        log_base = model_loglevel.intercept_ + model_loglevel.coef_[i]
    base_price = np.exp(log_base)
    next_price = np.exp(log_base + beta_bed_log)
    rows.append((borough, base_price, next_price, next_price - base_price))

dollar_table = pd.DataFrame(
    rows, columns=['borough', '0-BR base', '1-BR price', '$ per bedroom (0→1)']
).sort_values('$ per bedroom (0→1)', ascending=False)
print(f"Shared multiplier per bedroom: {multiplier:.3f}  (≈ {(multiplier-1)*100:.0f}% per bedroom)")
print()
print(dollar_table.round(0).to_string(index=False))

A bedroom in Manhattan adds many more dollars than a bedroom in the Bronx, even though both boroughs share the same *proportional* increase. The log model expresses that heterogeneity with a single extra coefficient instead of a fleet of interaction terms.

## Log transform on the response

In [ ]:
beta1_log = model_log_simple.coef_[0]
multiplier_simple = np.exp(beta1_log)
pct_change = (multiplier_simple - 1) * 100

print(f"log(price) ~ bedrooms:  β₁ = {beta1_log:.4f}")
print(f"Each bedroom multiplies price by e^{beta1_log:.4f} = {multiplier_simple:.3f}")
print(f"That corresponds to a {pct_change:.1f}% increase per bedroom.")

On the log scale, a one-unit increase in bedrooms multiplies the predicted price by $e^{\beta_1}$ — a fixed proportional jump, which becomes a larger dollar jump off a larger base.

## The four transform combinations

Depending on whether you log-transform $y$, $x$, or both, the coefficient takes on a different interpretation.

:::{.callout-important}
## Interpreting coefficients with log transforms

| Model | Coefficient $\beta_1$ means |
|-------|---------------------------|
| $y \sim x$ | a 1-unit increase in $x$ adds $\beta_1$ to $y$ |
| $\log(y) \sim x$ | a 1-unit increase in $x$ multiplies $y$ by $e^{\beta_1}$ |
| $y \sim \log(x)$ | a 1% increase in $x$ adds about $\beta_1/100$ to $y$ |
| $\log(y) \sim \log(x)$ | a 1% increase in $x$ gives about a $\beta_1$% change in $y$ |

: {.striped .hover}
:::

The first row is the standard linear model we've been using. The second row is the log-level model we just fit: each extra bedroom multiplies price by about $e^{0.29} \approx 1.33$. The bottom two rows — where $x$ itself appears on the log scale — are meant for features that span many orders of magnitude, so a *percentage* change is the natural unit (square footage, income, population, or in our data `accommodates`, which ranges over an order of magnitude).

:::{.callout-important}
## Definition: Elasticity

The coefficient on a log-log model is an **elasticity**: the percentage change in $y$ associated with a one-percent change in $x$. Elasticities are dimensionless, which makes them comparable across features on wildly different scales. Economists use them constantly — price elasticity of demand, income elasticity, wage elasticity — because "how much does revenue shift when price rises 1%" is a question that matters whether the price is \$5 or \$500.
:::

In [ ]:
# Log-log model: log(price) ~ log(accommodates)
log_accom = np.log(df['accommodates'])
model_loglog = LinearRegression().fit(log_accom.values.reshape(-1, 1), log_prices)
beta_loglog = model_loglog.coef_[0]

print(f"log(price) ~ log(accommodates):  β₁ = {beta_loglog:.3f}")
print(f"Elasticity: a 1% increase in accommodates → about {beta_loglog:.2f}% increase in price")
print(f"            a 10% larger listing → about {10 * beta_loglog:.1f}% higher price")

The elasticity lets us compare the "bedroom premium" to the "capacity premium" without worrying about whether one is measured in integers and the other in people — they live on the same percentage scale.

:::{.callout-tip}
## Think About It

When should the response be on a log scale? Prices, incomes, and populations are classic candidates — they grow multiplicatively, are bounded below by zero, and have right-skewed distributions. The log transform compresses the right tail and makes the distribution more symmetric, which also helps with heteroscedasticity (the fan-shaped residual pattern we examine in the diagnostics section below).
:::

# Diagnostics and dangers

With this many modeling choices at our disposal, we need tools to tell us when we've gone too far. Three diagnostics help: adjusted $R^2$ checks whether added features earn their place, multicollinearity warns when features duplicate each other, and residual plots reveal patterns the model missed.

## Adjusted $R^2$

Training $R^2$ never decreases when you add a feature — even a column of random noise. **Adjusted $R^2$** penalizes for the number of features:

$$R^2_{\text{adj}} = 1 - \frac{(1 - R^2)(n - 1)}{n - k - 1}$$

where $n$ is the number of observations and $k$ is the number of predictors. A useless feature increases $k$ without sufficiently reducing residual variance, so adjusted $R^2$ drops.

In [ ]:
def adjusted_r2(r2, n, k):
    return 1 - (1 - r2) * (n - 1) / (n - k - 1)

n = len(prices)
models_summary = {
    'bedrooms only': (model_simple.score(df[['bedrooms']], prices), 1),
    '+ bathrooms': (model_two.score(df[['bedrooms', 'bathrooms']], prices), 2),
    '+ room type': (r2_full, X_full.shape[1]),
    '+ borough + interactions': (r2_interact_full, X_interact_full.shape[1]),
}

print(f"{'Model':<30s}  {'R²':>6s}  {'Adj R²':>7s}  {'k':>3s}")
print("-" * 52)
for name, (r2, k) in models_summary.items():
    print(f"{name:<30s}  {r2:>.4f}  {adjusted_r2(r2, n, k):>.4f}  {k:>3d}")

For a more reliable assessment of model quality, we'll use held-out test data in [Chapter 6](lec06-validation.qmd).

## Multicollinearity

What happens when two features carry nearly the same information? The `beds` column (number of beds) is highly correlated with `bedrooms` — most listings have roughly one bed per bedroom.

In [ ]:
df_mc = df.dropna(subset=['beds']).copy()
corr_beds = df_mc['bedrooms'].corr(df_mc['beds'])
corr_bath = df_mc['bedrooms'].corr(df_mc['bathrooms'])

print(f"Correlation between bedrooms and beds:      {corr_beds:.3f}")
print(f"Correlation between bedrooms and bathrooms:  {corr_bath:.3f}")

A correlation of 0.9 between `bedrooms` and `beds` is enormous — these two columns carry almost the same information. The correlation between `bedrooms` and `bathrooms` is far weaker by comparison. Plotting the `bedrooms`/`beds` cloud and the three-way correlation matrix makes the near-duplication visible.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2),
                         gridspec_kw={'width_ratios': [1.15, 1]})

# Left: 2D histogram
h = axes[0].hist2d(
    df_mc['bedrooms'], df_mc['beds'],
    bins=[np.arange(-0.5, 7), np.arange(-0.5, 10)],
    cmap='BuPu', cmin=1,
    norm=mcolors.LogNorm(vmin=1, vmax=20000),
    edgecolors='white', linewidths=0.5)
cb = plt.colorbar(h[3], ax=axes[0], shrink=0.82, pad=0.02)
cb.set_label('Number of listings', fontsize=10)
axes[0].plot([0, 6], [0, 6], color='#d62728', ls='--', lw=2.2,
             label='beds = bedrooms', zorder=5)
axes[0].set_xlabel('Bedrooms')
axes[0].set_ylabel('Beds')
axes[0].set_title('Most listings cluster near the diagonal')
axes[0].legend(fontsize=9)

# Right: correlation heatmap
corr = df_mc[['bedrooms', 'bathrooms', 'beds']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, ax=axes[1], square=True,
            linewidths=2, linecolor='white',
            annot_kws={'fontsize': 14, 'fontweight': 'bold'},
            cbar_kws={'shrink': 0.82, 'label': 'Correlation'})
axes[1].set_title('Feature correlations')

plt.tight_layout(w_pad=3)
plt.show()

When two feature vectors point in nearly the same direction, adding the second one barely expands the span — a phenomenon with a name.

:::{.callout-important}
## Definition: Multicollinearity

**Multicollinearity** occurs when two or more feature columns are highly correlated — their vectors point in nearly the same direction. The span barely grows from the additional feature, and the normal equations become ill-conditioned: $(X^TX)^{-1}$ amplifies small perturbations in the data.
:::

In [ ]:
# Compare models with and without beds
X_no_beds = df_mc[['bedrooms', 'bathrooms']].astype(float)
X_with_beds = df_mc[['bedrooms', 'bathrooms', 'beds']].astype(float)
y_mc = df_mc['price']

m_no = LinearRegression().fit(X_no_beds, y_mc)
m_with = LinearRegression().fit(X_with_beds, y_mc)

print("Without beds:")
for name, coef in zip(X_no_beds.columns, m_no.coef_):
    print(f"  {name:12s}  {coef:+.2f}")
print(f"  R² = {m_no.score(X_no_beds, y_mc):.4f}")

print("\nWith beds:")
for name, coef in zip(X_with_beds.columns, m_with.coef_):
    print(f"  {name:12s}  {coef:+.2f}")
print(f"  R² = {m_with.score(X_with_beds, y_mc):.4f}")

Watch what happens to the coefficients. Without beds, bedrooms gets about \$46 and bathrooms about \$12. Once beds enters, bedrooms drops to roughly \$20, bathrooms drops to about \$5, and beds picks up around \$29. $R^2$ improves, but the bedrooms coefficient has been cut by more than half — the model is redistributing credit among correlated features, and the individual numbers have lost their stable meaning. Predictions remain serviceable; interpretations do not.

## Residual diagnostics

The normal equations guarantee $X^T \epsilon = 0$ — no linear pattern remains between features and residuals. Patterns can still lurk. A scatter plot of residuals vs. predicted values reveals them: fan shapes, curves, or clusters.

In [ ]:
y_hat_full = model_full.predict(X_full)
residuals_full = prices.values - y_hat_full

def rolling_envelope(x, r, n_bins=25):
    """±2σ envelope of residuals r across fitted values x, computed in bins."""
    order = np.argsort(x)
    x_sorted, r_sorted = x[order], r[order]
    edges = np.quantile(x_sorted, np.linspace(0, 1, n_bins + 1))
    centers, bands = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (x_sorted >= lo) & (x_sorted <= hi)
        if mask.sum() >= 20:
            centers.append(x_sorted[mask].mean())
            bands.append(2 * r_sorted[mask].std())
    return np.array(centers), np.array(bands)

fig, ax = plt.subplots(figsize=(7, 4))
mean_r = residuals_full.mean()
median_r = np.median(residuals_full)
ax.hist(residuals_full, bins=60, edgecolor='white', alpha=0.7)
ax.axvline(median_r, color='#2171b5', lw=2, ls='--', label=f'median ≈ {median_r:.0f}')
ax.axvline(mean_r, color='#d62728', lw=2, ls='--', label=f'mean ≈ {mean_r:.0f}')
ax.set_xlabel('Residual ($)')
ax.set_ylabel('Count')
ax.set_title('Distribution of residuals (right-skewed)')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

The distribution is right-skewed — the mean sits above the median, the sign of a long upper tail. The asymmetry hints at a second problem we'll see below: the *spread* of residuals is not constant across fitted values.

## Heteroscedasticity

When the variance of the residuals changes with the predicted value, we have **heteroscedasticity** (from the Greek: "different scatter"). A *fan-shaped* residual plot is the classic signal: if you trace the outer edge of the residual cloud from left to right, it looks like a megaphone opening up — narrow where predictions are small, wide where they're large.

:::{.callout-important}
## Definition: Heteroscedasticity

**Heteroscedasticity** means the spread of residuals is not constant across the range of predictions. A fan-shaped residual plot is the telltale sign. Coefficient estimates remain unbiased, but standard errors (and therefore confidence intervals and p-values) become unreliable under heteroscedasticity.
:::

The log transform often helps: by compressing the right tail of the price distribution, it stabilizes the variance of the residuals.

In [ ]:
# Compare residual plots: level vs log. Both panels get the ±2σ envelope
# so the "fan" is visible rather than inferred.
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4))

# Level model
axes[0].scatter(y_hat_full, residuals_full, alpha=0.05, s=5)
axes[0].axhline(0, color='red', lw=1.5)
c, b = rolling_envelope(y_hat_full, residuals_full)
axes[0].plot(c, b, color='red', lw=1.5, label='±2σ envelope')
axes[0].plot(c, -b, color='red', lw=1.5)
axes[0].legend(loc='upper left', fontsize=9)
axes[0].set_xlabel('Predicted price ($)')
axes[0].set_ylabel('Residual ($)')
axes[0].set_title('Level model: fan-shaped residuals')

# Log model
model_log_full = LinearRegression().fit(X_full, np.log(prices))
log_resid = np.log(prices.values) - model_log_full.predict(X_full)
log_fitted = model_log_full.predict(X_full)
axes[1].scatter(log_fitted, log_resid, alpha=0.05, s=5)
axes[1].axhline(0, color='red', lw=1.5)
c_log, b_log = rolling_envelope(log_fitted, log_resid)
axes[1].plot(c_log, b_log, color='red', lw=1.5, label='±2σ envelope')
axes[1].plot(c_log, -b_log, color='red', lw=1.5)
axes[1].legend(loc='upper left', fontsize=9)
axes[1].set_xlabel('Predicted log(price)')
axes[1].set_ylabel('Residual')
axes[1].set_title('Log model: more uniform spread')

plt.tight_layout()
plt.show()

The log-transformed model produces residuals with more uniform spread — the fan shape is reduced.

# Recap: the regression toolkit

This chapter extended the regression framework from Chapter 4 in three directions: multiple regression with numeric features, one-hot encoding for categorical variables, and feature engineering — polynomials, interactions, missingness indicators, and log transforms. Each move expands what "linear in parameters" can express. The $R^2$ progression from bedrooms alone through borough interactions shows how much pricing signal each move unlocks.

## What we still can't answer

Three questions remain open:

1. **Is the coefficient real or noise?** The bathrooms coefficient is about \$22, but with a different sample it might be \$18 or \$26. We'll quantify the uncertainty with confidence intervals in [Chapter 12](lec12-regression-inference.qmd).
2. **Does the model generalize to new data?** Training $R^2$ always increases with more features, even noise features. We'll develop train-test splits and cross-validation in [Chapter 6](lec06-validation.qmd).
3. **Does adding a bathroom *cause* higher prices?** Regression coefficients measure association. Causation requires a different framework — [Chapter 18](lec18-causal.qmd).

## Study guide

### Key ideas

- **Normal equations** — $X^TX\beta = X^Ty$. The algebraic form of "the residual is orthogonal to every feature." Derived by stacking the orthogonality conditions. More independent feature columns means a larger span and a closer projection of $y$.
- **Coefficient interpretation** — in multiple regression, each coefficient measures the association between a feature and the response **holding all other features constant**. Coefficients change between simple and multiple regression when the features are correlated.
- **One-hot encoding** — converts a categorical variable with $k$ categories into $k-1$ binary columns. The dropped category is the **reference level**, absorbed into the intercept.
- **Feature engineering** — creating new input columns (polynomials, interactions, indicators) so a linear model can capture nonlinear patterns. The model stays linear in $\beta$.
- **Polynomial features** — powers of a variable ($x^2, x^3, \ldots$) that capture curvature. High-degree polynomials overfit and extrapolate wildly.
- **Interaction terms** — products of two features ($x_1 \times x_2$) that let the association between one feature and the response depend on another.
- **Log transform** — regressing $\log(y)$ on $x$ gives multiplicative coefficients (percentage changes per unit increase in $x$). Appropriate for positive, right-skewed responses.
- **Adjusted $R^2$** — penalizes for the number of features, preventing $R^2$ from rewarding useless predictors.
- **Multicollinearity** — when feature columns are highly correlated, the span barely grows. Coefficients become unstable and hard to interpret.
- **Heteroscedasticity** — non-constant residual variance (fan-shaped residual plots). Coefficients remain unbiased but standard errors become unreliable.
- **Missing values as features** — pair imputed values with a binary missingness indicator to let the model learn from informative missingness.

### Key formulas

- **Normal equations**: $X^T X \beta = X^T y$, with closed-form solution $\beta = (X^T X)^{-1} X^T y$ when $X^T X$ is invertible.
- **Orthogonality of residuals**: $X^T \epsilon = 0$ where $\epsilon = y - X\beta$.
- **Adjusted $R^2$**: $R^2_{\text{adj}} = 1 - \dfrac{(1 - R^2)(n - 1)}{n - k - 1}$, where $n$ is the sample size and $k$ is the number of predictors.
- **Log-level coefficient**: if $\log y = \beta_0 + \beta_1 x$, then a one-unit change in $x$ multiplies predicted $y$ by $e^{\beta_1}$.

### Computational tools

- `np.linalg.solve(A, b)` — solve $Ax = b$ without forming $A^{-1}$ (for the normal equations)
- `pd.get_dummies(df, drop_first=True)` — one-hot encode categorical variables, dropping the reference level
- `PolynomialFeatures(degree=d)` — generate polynomial features up to degree $d$
- `LinearRegression().fit(X, y)` — fit a linear model; `.coef_` for slopes, `.intercept_` for intercept
- `.score(X, y)` — compute $R^2$ on data
- `.dropna(subset=[...])` — drop rows with missing values in specified columns
- `.fillna(value)` — fill missing values (e.g., with the median for imputation)
- `.groupby(col)` — split a DataFrame by a categorical column for per-group computation

### For the quiz

- Be able to write and interpret the normal equations ($X^T \epsilon = 0$).
- Given a regression output with dummy variables, interpret coefficients relative to the reference level.
- Explain why the bedrooms coefficient changes between simple and multiple regression.
- Know what polynomial overfitting looks like and why training $R^2$ is misleading.
- Interpret a coefficient from a log-transformed model (percentage change vs. dollar change).
- Identify heteroscedasticity from a residual plot and suggest a remedy.